# IDA Results Analysis and Interpretation

**Notebook:** 03_ida_results_analysis.ipynb

**Purpose:** Analyze incremental dynamic analysis (IDA) results and prepare data for ML training

**Contents:**
1. Load and validate IDA results
2. Compute IDA curves for all building-GM combinations
3. Extract peak inter-story drift ratios (PIDR)
4. Perform feature engineering
5. Analyze framework performance comparison
6. Prepare datasets for ML training

## 1. Environment Setup and Load IDA Results

In [ ]:
# Import libraries
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import yaml
import logging
from scipy import stats

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
%matplotlib inline

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('ida_analysis')

project_root = Path('..').resolve()
sys.path.insert(0, str(project_root))

# Load config
with open(project_root / 'config' / 'analysis_config.yaml') as f:
    analysis_config = yaml.safe_load(f)

with open(project_root / 'config' / 'bnbc_parameters.yaml') as f:
    bnbc_params = yaml.safe_load(f)

logger.info('✓ Environment ready')

In [ ]:
# Create synthetic IDA results (in production, these would come from actual OpenSeesPy analysis)
np.random.seed(42)

buildings = {
    'b5_nonsway': {'n_stories': 5, 'framework': 'nonsway', 'period': 0.45},
    'b5_omrf': {'n_stories': 5, 'framework': 'omrf', 'period': 0.50},
    'b5_imrf': {'n_stories': 5, 'framework': 'imrf', 'period': 0.52},
    'b5_smrf': {'n_stories': 5, 'framework': 'smrf', 'period': 0.55},
    'b7_smrf': {'n_stories': 7, 'framework': 'smrf', 'period': 0.70},
    'b10_smrf': {'n_stories': 10, 'framework': 'smrf', 'period': 1.00},
}

zones = [1, 2, 3, 4]
gm_count = 10  # 10 ground motions per zone
sa_range = np.arange(0.1, 1.6, 0.1)

# Generate synthetic IDA results
ida_results = []

for building_id, building_props in buildings.items():
    for zone in zones:
        for gm_idx in range(gm_count):
            for sa in sa_range:
                # Simulate increasing drift with intensity (realistic behavior)
                base_drift = 0.002 + 0.015 * (sa ** 1.5) * np.random.normal(1, 0.1)
                
                # Framework effect
                fw_factor = {'nonsway': 1.3, 'omrf': 1.0, 'imrf': 0.85, 'smrf': 0.75}[building_props['framework']]
                pidr = max(0.001, base_drift * fw_factor)
                
                ida_results.append({
                    'building_id': building_id,
                    'n_stories': building_props['n_stories'],
                    'framework': building_props['framework'],
                    'period': building_props['period'],
                    'zone': zone,
                    'gm_id': f'gm_z{zone}_{gm_idx:03d}',
                    'sa_intensity': sa,
                    'pidr': pidr,
                    'max_accel': sa * 0.9
                })

df_ida = pd.DataFrame(ida_results)
print(f'Generated {len(df_ida)} IDA results')
print(f'\nDataFrame shape: {df_ida.shape}')
print(f'\nSample IDA results:')
print(df_ida.head(10))

## 2. Compute and Visualize IDA Curves

In [ ]:
# Compute IDA curves for each building-zone combination
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

building_list = list(buildings.keys())
colors = plt.cm.RdYlGn_r(np.linspace(0, 1, len(zones)))

for idx, building_id in enumerate(building_list):
    ax = axes[idx]
    
    for zone_idx, zone in enumerate(zones):
        # Get data for this building-zone
        subset = df_ida[(df_ida['building_id'] == building_id) & (df_ida['zone'] == zone)]
        
        # Group by ground motion and compute median/percentile
        grouped = subset.groupby('sa_intensity')['pidr'].agg(['median', 'std', 'count']).reset_index()
        
        # Plot median with confidence band
        ax.plot(grouped['sa_intensity'], grouped['median'], 'o-', linewidth=2, 
                label=f'Zone {zone}', color=colors[zone_idx], markersize=6)
        
        # Add uncertainty band
        upper = grouped['median'] + grouped['std']
        lower = grouped['median'] - grouped['std']
        ax.fill_between(grouped['sa_intensity'], lower, upper, alpha=0.2, color=colors[zone_idx])
    
    ax.set_xlabel('Spectral Acceleration Sa(T) (g)', fontsize=10, fontweight='bold')
    ax.set_ylabel('Peak Inter-Story Drift Ratio', fontsize=10, fontweight='bold')
    ax.set_title(f'{building_id.upper()}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9, loc='upper left')
    ax.set_ylim([0, 0.15])

plt.suptitle('IDA Curves by Building and Seismic Zone', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig(project_root / 'results' / 'figures' / 'ida_curves_by_zone.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved: ida_curves_by_zone.png')

## 3. Extract Key IDA Parameters

In [ ]:
# Define performance levels based on PIDR
performance_levels = {
    'IO': 0.01,   # Immediate Occupancy
    'LS': 0.02,   # Life Safety
    'CP': 0.05    # Collapse Prevention
}

# Extract intensity at each performance level
ida_metrics = []

for building_id in buildings.keys():
    for zone in zones:
        subset = df_ida[(df_ida['building_id'] == building_id) & (df_ida['zone'] == zone)]
        
        # Get median curve
        grouped = subset.groupby('sa_intensity')['pidr'].median().reset_index()
        grouped = grouped.sort_values('sa_intensity')
        
        metric_row = {
            'building_id': building_id,
            'zone': zone,
            'n_stories': buildings[building_id]['n_stories'],
            'framework': buildings[building_id]['framework'],
            'period': buildings[building_id]['period']
        }
        
        # Interpolate Sa at each performance level
        from scipy.interpolate import interp1d
        if len(grouped) > 1:
            f = interp1d(grouped['pidr'], grouped['sa_intensity'], 
                         bounds_error=False, fill_value='extrapolate')
            
            for level, pidr_threshold in performance_levels.items():
                sa_at_level = float(f(pidr_threshold))
                metric_row[f'Sa_{level}'] = max(0.05, min(2.0, sa_at_level))  # Bounds
        
        ida_metrics.append(metric_row)

df_metrics = pd.DataFrame(ida_metrics)

print('IDA Metrics - Spectral Acceleration at Performance Levels:')
print(df_metrics[['building_id', 'zone', 'Sa_IO', 'Sa_LS', 'Sa_CP']].head(12).to_string())

# Save metrics
df_metrics.to_csv(project_root / 'data' / 'processed' / 'ida_metrics.csv', index=False)
print('\n✓ Saved: ida_metrics.csv')

## 4. Framework Comparison Analysis

In [ ]:
# Compare frameworks at 5-story level
b5_buildings = [b for b in buildings.keys() if 'b5' in b]
frameworks = ['nonsway', 'omrf', 'imrf', 'smrf']

fig, axes = plt.subplots(1, len(zones), figsize=(16, 4))

for zone_idx, zone in enumerate(zones):
    ax = axes[zone_idx]
    
    for fw_idx, framework in enumerate(frameworks):
        building_id = f'b5_{framework}'
        subset = df_ida[(df_ida['building_id'] == building_id) & (df_ida['zone'] == zone)]
        
        grouped = subset.groupby('sa_intensity')['pidr'].median().reset_index()
        
        ax.plot(grouped['sa_intensity'], grouped['pidr'], 'o-', linewidth=2.5,
                label=framework.upper(), markersize=6)
    
    ax.set_xlabel('Sa(T) (g)', fontsize=11, fontweight='bold')
    ax.set_ylabel('PIDR', fontsize=11, fontweight='bold')
    ax.set_title(f'Zone {zone}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 0.12])
    
    # Add performance level lines
    for level, pidr_val in performance_levels.items():
        ax.axhline(pidr_val, linestyle='--', alpha=0.5, linewidth=1, color='gray')
        if zone == 0:
            ax.text(0.1, pidr_val + 0.003, level, fontsize=9, style='italic')
    
    if zone == 0:
        ax.legend(fontsize=10, loc='upper left')

plt.suptitle('Framework Performance Comparison - 5-Story Building', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(project_root / 'results' / 'figures' / 'framework_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved: framework_comparison.png')

In [ ]:
# Quantitative framework comparison
comparison_data = []

for zone in zones:
    zone_data = df_ida[df_ida['zone'] == zone]
    
    for framework in frameworks:
        fw_data = zone_data[zone_data['framework'] == framework]
        
        comparison_data.append({
            'Zone': zone,
            'Framework': framework.upper(),
            'Avg PIDR': fw_data['pidr'].mean(),
            'Median PIDR': fw_data['pidr'].median(),
            'Max PIDR': fw_data['pidr'].max(),
            'Std PIDR': fw_data['pidr'].std(),
            'n_records': len(fw_data)
        })

df_comparison = pd.DataFrame(comparison_data)
print('\nFramework Performance Statistics:') print(df_comparison.to_string(index=False))

## 5. Feature Engineering for ML

In [ ]:
# Create feature matrix for ML training
df_ml = df_ida.copy()

# Add BNBC zone characteristics
zone_map = {}
for zone in zones:
    zone_key = f'zone_{zone}'
    zone_map[zone] = {
        'pga': bnbc_params['seismic_zones'][zone_key]['pga'],
        'z_coeff': bnbc_params['seismic_zones'][zone_key]['z_coeff']
    }

df_ml['pga_bnbc'] = df_ml['zone'].map(lambda z: zone_map[z]['pga'])
df_ml['z_coeff'] = df_ml['zone'].map(lambda z: zone_map[z]['z_coeff'])

# Encode categorical features
from sklearn.preprocessing import LabelEncoder

le_fw = LabelEncoder()
df_ml['framework_code'] = le_fw.fit_transform(df_ml['framework'])

# Create interaction features
df_ml['period_sa'] = df_ml['period'] * df_ml['sa_intensity']
df_ml['period_pga'] = df_ml['period'] * df_ml['pga_bnbc']
df_ml['sa_pga_ratio'] = df_ml['sa_intensity'] / df_ml['pga_bnbc']

# Feature list for ML
feature_cols = ['n_stories', 'period', 'framework_code', 'zone', 
                'sa_intensity', 'pga_bnbc', 'z_coeff', 
                'period_sa', 'period_pga', 'sa_pga_ratio']

print(f'Features for ML training: {len(feature_cols)}')
print(f'Feature columns:\n  {feature_cols}')
print(f'\nTarget variable: pidr')
print(f'\nML Dataset shape: {df_ml.shape}')
print(f'\nSample with features:')
print(df_ml[feature_cols + ['pidr']].head(5))

In [ ]:
# Data quality checks
print('Data Quality Assessment:')
print('=' * 60)

print(f'\nMissing values:')
missing = df_ml[feature_cols + ['pidr']].isnull().sum()
if missing.sum() == 0:
    print('  ✓ No missing values')
else:
    print(missing[missing > 0])

print(f'\nTarget variable (PIDR) distribution:')
print(f'  Mean: {df_ml["pidr"].mean():.4f}')
print(f'  Median: {df_ml["pidr"].median():.4f}')
print(f'  Min: {df_ml["pidr"].min():.4f}')
print(f'  Max: {df_ml["pidr"].max():.4f}')
print(f'  Std: {df_ml["pidr"].std():.4f}')
print(f'  Skewness: {df_ml["pidr"].skew():.3f}')
print(f'  Kurtosis: {df_ml["pidr"].kurtosis():.3f}')

print(f'\nFeature ranges:')
for col in ['n_stories', 'period', 'sa_intensity']:
    print(f'  {col}: [{df_ml[col].min():.2f}, {df_ml[col].max():.2f}]')

print(f'\nClass balance:')
print(f'  Records per framework:')
for fw in frameworks:
    count = len(df_ml[df_ml['framework'] == fw])
    pct = count / len(df_ml) * 100
    print(f'    {fw.upper()}: {count} ({pct:.1f}%)')

## 6. Export Processed Data

In [ ]:
# Split into train and test sets
from sklearn.model_selection import train_test_split

X = df_ml[feature_cols]
y = df_ml['pidr']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=df_ml['framework']
)

# Create combined dataframes
train_data = X_train.copy()
train_data['pidr'] = y_train
train_data['split'] = 'train'

test_data = X_test.copy()
test_data['pidr'] = y_test
test_data['split'] = 'test'

# Export datasets
train_data.to_csv(project_root / 'data' / 'processed' / 'training_data.csv', index=False)
test_data.to_csv(project_root / 'data' / 'processed' / 'test_data.csv', index=False)
df_ml[feature_cols + ['pidr']].to_csv(project_root / 'data' / 'processed' / 'ida_results.csv', index=False)

print(f'✓ Exported datasets:')
print(f'  - Training: {len(train_data)} records')
print(f'  - Test: {len(test_data)} records')
print(f'  - Total: {len(df_ml)} records')
print(f'  - Features: {len(feature_cols)}')
print(f'\nFiles saved:')
print(f'  - data/processed/training_data.csv')
print(f'  - data/processed/test_data.csv')
print(f'  - data/processed/ida_results.csv')

In [ ]:
# Summary statistics
print('='*60)
print('IDA ANALYSIS SUMMARY')
print('='*60)
print(f'\nDataset Summary:')
print(f'  Total Records: {len(df_ida):,}')
print(f'  Buildings Analyzed: {df_ida["building_id"].nunique()}')
print(f'  Seismic Zones: {df_ida["zone"].nunique()}')
print(f'  Frameworks: {df_ida["framework"].nunique()}')
print(f'\nIntensity Measure Range:')
print(f'  Sa(T): {df_ida["sa_intensity"].min():.2f}g to {df_ida["sa_intensity"].max():.2f}g')
print(f'\nPeak Inter-Story Drift:')
print(f'  Mean: {df_ida["pidr"].mean():.4f}')
print(f'  Range: {df_ida["pidr"].min():.4f} to {df_ida["pidr"].max():.4f}')
print(f'\nML Training Data:')
print(f'  Features: {len(feature_cols)}')
print(f'  Training samples: {len(train_data)}')
print(f'  Test samples: {len(test_data)}')
print(f'  Train/Test split: {len(train_data)/len(df_ml)*100:.1f}% / {len(test_data)/len(df_ml)*100:.1f}%')
print('\n' + '='*60)
print('✓ IDA analysis complete and ready for ML training!')
print('='*60)